# Energy Fraud Detection — Runner

Works in **Google Colab** and **VS Code** (local kernel or connected Colab runtime).

If using Colab in the browser, set the runtime to **GPU** first: `Runtime → Change runtime type → T4 GPU`.

## Step 1 — Environment setup

Detects whether you are in Colab or running locally in VS Code and sets paths accordingly.

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Clone the repo (Colab session starts empty)
    os.system('git clone https://github.com/Vict5/smart-meter-fraud.git')
    REPO_PATH = '/content/smart-meter-fraud'
else:
    # Running locally in VS Code — repo is already on disk.
    # Update this to the actual path if needed.
    REPO_PATH = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f'Environment : {"Colab" if IN_COLAB else "Local / VS Code"}')
print(f'Working dir : {os.getcwd()}')

## Step 2 — Install dependencies

**Colab**: run every session (~3 min).  
**VS Code / local**: run once; skip afterwards if your environment already has them.

In [ ]:
%%capture
!pip install \
    'scikit-learn==1.3.2' \
    'imbalanced-learn==0.11.0' \
    polars pyarrow tqdm \
    pytorch-lightning torchmetrics \
    xgboost shap

In [ ]:
print('Installation complete.')

## Step 3 — Provide raw data files

The 6 raw CSV files are needed: `CONSUMI.csv`, `ANAGRAFICA.csv`, `LAVORI.csv`, `INTERRUZIONI.csv`, `PAROLE_DI_STATO.csv`, `LABELS.csv`.

- **Colab**: the cell below opens a file-picker to upload them.
- **VS Code / local**: place the files in `data/raw/` manually and skip this cell.

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)

if IN_COLAB:
    from google.colab import files
    print('Select all 6 raw CSV files in the upload dialog...')
    uploaded = files.upload()
    for filename, data in uploaded.items():
        dest = os.path.join('data/raw', filename)
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  saved -> {dest}')
else:
    print('Local mode: place raw CSV files in data/raw/ then continue.')

print('\nFiles in data/raw/:', os.listdir('data/raw'))

## Step 4 — Preprocess raw data

Reads from `data/raw/` and writes cleaned CSV files to `data/processed/`.  
Skip if `data/processed/` already contains the 6 files.

In [ ]:
import os, pathlib

processed = pathlib.Path('data/processed')
already_done = processed.exists() and len(list(processed.glob('*.csv'))) == 6

if already_done:
    print('data/processed/ already has 6 files, skipping preprocessing.')
else:
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('src/models/scalers', exist_ok=True)

    from src.features.build_features import PreprocessDataset
    preprocessor = PreprocessDataset(pathlib.Path('data/raw'))
    preprocessor.preprocess_all()
    print('Preprocessing done. Files in data/processed/:', os.listdir('data/processed'))

## Step 5 — (Optional) Restore cached models

Skip on the first run. On later **Colab** sessions, uncomment and run to restore
previously trained models and avoid ~1 hour of autoencoder retraining.  
**VS Code / local**: models persist on disk between runs automatically — skip this cell.

In [ ]:
# Colab only — restore from Drive
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree(f'{CACHE}/models',              'models',                   dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/embeddings',          'data/embeddings',          dirs_exist_ok=True)
# shutil.copytree(f'{CACHE}/combined_embeddings', 'data/combined_embeddings', dirs_exist_ok=True)
# print('Models restored from Drive.')

## Step 6 — Enable inline plots

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

## Step 7 — Run the pipeline

First run trains all 5 autoencoders (~1 hr on GPU, ~several hrs on CPU).  
Later runs detect saved files and jump straight to classification.

| Mode | Description |
|------|-------------|
| `XGBoost` | 4-fold CV XGBoost ensemble (recommended) |
| `RandomForest` | 4-fold CV Random Forest ensemble |
| `DNN_Classifier` | 4-fold CV MLP ensemble |
| `Binary_Classifier` | Threshold anomaly detection (requires `ONLY_REGULAR=True` in config) |

In [ ]:
from src.pipeline import main
main(mode='XGBoost')

In [ ]:
# Other modes:
# main(mode='RandomForest')
# main(mode='DNN_Classifier')
# main(mode='Binary_Classifier')  # only when ONLY_REGULAR=True in config

## Step 8 — (Optional) Save trained models to Drive

**Colab only** — run after the first successful pipeline run to cache models for future sessions.  
**VS Code / local**: skip — models are already saved to disk.

In [ ]:
# Colab only — save to Drive
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# CACHE = '/content/drive/MyDrive/smart-meter-fraud-cache'
# shutil.copytree('models',                   f'{CACHE}/models',              dirs_exist_ok=True)
# shutil.copytree('data/embeddings',          f'{CACHE}/embeddings',          dirs_exist_ok=True)
# shutil.copytree('data/combined_embeddings', f'{CACHE}/combined_embeddings',  dirs_exist_ok=True)
# print('Models saved to Drive.')

## (Optional) Override config hyperparameters

In [ ]:
# Quick smoke-test with fewer epochs — run this BEFORE Step 7
# from src.models.config import Config
# Config.N_EPOCHS_AE = 50
# Config.N_EPOCHS_DNN = 100